# NLPG Week 3 Practical
## Representing text as tokens, lemmas and bag-of-words vectors

This practical follows one question through several computational representations:

> **What information will the model actually receive?**

You will compare tokenisation choices, inspect stemming and lemmatisation, construct bag-of-words vectors, and compare documents with cosine similarity.

### During the practical

- Run the cells in order.
- You are not required to submit answers or edit this notebook to record them.
- Take your own notes on the three required activities and what you observe.
- Keep those notes and bring them to this week's summing-up lecture, where we may compare and discuss your findings.

Expected time for the required work: **about 85–100 minutes**.


## Learning outcomes

By the end of the practical, you should be able to:

- explain why different tokenisers produce different evidence;
- distinguish a stem from a lemma;
- construct and read a document–term matrix;
- explain what a unigram bag-of-words representation preserves and discards;
- interpret cosine similarity as similarity under a particular representation.

The practical uses **NLTK** for tokenisation and morphology, and **scikit-learn** for bag-of-words vectors and cosine similarity.


## Setup

Run this cell once. It imports the required libraries and downloads the small WordNet data files used by NLTK's lemmatiser. Colab normally already contains the Python packages; the installation step runs only if something is missing.


In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

required_packages = {
    "nltk": "nltk",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "seaborn": "seaborn",
    "matplotlib": "matplotlib",
}

missing = [package for module, package in required_packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call(["uv", "add", "-q", *missing])

import re
import nltk
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from nltk.tokenize import TreebankWordTokenizer, wordpunct_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import os
from dotenv import load_dotenv

load_dotenv()

for resource in ("wordnet", "omw-1.4"):
    nltk.download(resource, download_dir=os.environ["NLTK_DATA"], quiet=True)

pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="white", font_scale=1.0)
print("Setup complete.")


# Required activity 1 — Tokenisation changes the evidence

Before running the next cell, look at this sentence and predict where each method might place token boundaries:

> York-based students paid £12.50 — didn’t they?

The methods are deliberately simple. The purpose is not to identify a universally correct tokenisation, but to make the choices visible.


In [ ]:
sample_text = "York-based students paid £12.50 — didn’t they?"
treebank = TreebankWordTokenizer()

tokenisations = {
    "Characters": list(sample_text),
    "Whitespace split": sample_text.split(),
    "Word/punctuation": wordpunct_tokenize(sample_text),
    "Treebank": treebank.tokenize(sample_text),
    "Treebank after normalising apostrophe: ’ -> '": treebank.tokenize(sample_text.replace("’", "'")),
}

rows = []
for method, tokens in tokenisations.items():
    rows.append({
        "method": method,
        "tokens": len(tokens),
        "types": len(set(tokens)),
        "result": " | ".join(tokens),
    })

pd.DataFrame(rows)


### What to notice

Make a note of:

- what happens to **York-based**, **£12.50**, **didn’t** and the final question mark;
- why changing the apostrophe before tokenisation changes the Treebank result;
- which counts would change if these outputs were used to train a later model.

The original character sequence is the same linguistic material, but each tokeniser supplies different observable events.


# Required activity 2 — Stems and lemmas make different generalisations

A **stemmer** applies mechanical rules intended to bring related strings together. Its output need not be a dictionary word.

A **lemmatiser** attempts to return a dictionary headword or **lemma**. It may need grammatical category information and access to a lexicon.

The codes used below are `n` for noun, `v` for verb, `a` for adjective and `r` for adverb.


In [ ]:
stemmer = PorterStemmer()
lemmatiser = WordNetLemmatizer()

examples = [
    ("cats", "n"),
    ("studies", "n"),
    ("meeting", "n"),
    ("meeting", "v"),
    ("left", "n"),
    ("left", "v"),
    ("went", "v"),
    ("better", "a"),
    ("better", "r"),
    ("happiness", "n"),
    ("wugs", "n"),
]

comparison = pd.DataFrame([
    {
        "word form": form,
        "category supplied": pos,
        "Porter stem": stemmer.stem(form),
        "WordNet lemma": lemmatiser.lemmatize(form, pos),
    }
    for form, pos in examples
])

comparison


### What to notice

Make a note of:

- outputs such as **studi** and **happi**, which are useful stems but not ordinary dictionary forms;
- forms whose lemma changes when a different grammatical category is supplied;
- the treatment of irregular forms such as **went**, **better** and **left**;
- what happens to **wugs**.

**Wug** is an established harmless non-word used in a classic test of morphological productivity. A speaker can readily form **wugs**, even if a lexicon-based lemmatiser does not contain **wug**. This is a useful reminder that productive language knowledge is not simply a list of previously stored words.


## A transparent normalisation for the next activity

Automatic lemmatisation of unrestricted text normally requires more machinery, including grammatical category decisions. To keep the bag-of-words exercise focused, the next cell provides a small, visible lemma table for the words in our example documents.

This table is teaching scaffolding, not a general English lemmatiser.


In [ ]:
lemma_lookup = {
    "dogs": "dog", "cats": "cat",
    "chased": "chase", "chasing": "chase",
    "gardens": "garden",
    "markets": "market",
    "rose": "rise", "rises": "rise", "rising": "rise",
    "investors": "investor",
    "bought": "buy", "buys": "buy", "buying": "buy",
    "shares": "share",
    "were": "be", "are": "be",
}

def simple_word_tokens(text):
    # Lower-case alphabetic tokens, retaining internal apostrophes.
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower())

def supplied_lemmatisation(text):
    tokens = simple_word_tokens(text)
    return [lemma_lookup.get(token, token) for token in tokens]

demo = "Dogs were chasing cats through the gardens."
print("Tokens:", simple_word_tokens(demo))
print("Lemmas:", supplied_lemmatisation(demo))


# Required activity 3 — Bag of words replaces sequence with geometry

A unigram **bag of words** represents a document by counting vocabulary items. Each vocabulary item defines one coordinate of a vector.

The vector has a fixed order of coordinates, but it does **not** record the order in which words occurred in the document.

We will begin with three tiny documents so that the complete representation remains visible.


In [ ]:
tiny_labels = ["D1", "D2", "D3"]
tiny_documents = [
    "The dog chased the cat.",
    "The cat chased the dog.",
    "The dog slept.",
]

tiny_vectoriser = CountVectorizer(token_pattern=r"(?u)\b[a-zA-Z]+\b")
tiny_matrix = tiny_vectoriser.fit_transform(tiny_documents)
tiny_vocabulary = list(tiny_vectoriser.get_feature_names_out())

print("Vocabulary order:", tiny_vocabulary)
pd.DataFrame(tiny_matrix.toarray(), index=tiny_labels, columns=tiny_vocabulary)


Documents D1 and D2 contain the same vocabulary items with the same counts. Their bag-of-words vectors are therefore identical, although the dog and cat exchange grammatical roles.

Now calculate **cosine similarity**:

$$
\operatorname{cosine}(d_1,d_2)
=
\frac{d_1\cdot d_2}{\lVert d_1\rVert\lVert d_2\rVert}
$$

Cosine similarity is 1 when two non-zero vectors point in the same direction, and 0 when these non-negative count vectors share no active dimensions.


In [ ]:
tiny_similarity = cosine_similarity(tiny_matrix)
tiny_similarity_table = pd.DataFrame(
    tiny_similarity, index=tiny_labels, columns=tiny_labels
)

fig, ax = plt.subplots(figsize=(5.8, 4.7))
sns.heatmap(
    tiny_similarity_table,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Cosine similarity of tiny bag-of-words vectors")
plt.tight_layout()
plt.show()


### What to notice

Make a note of why D1 and D2 receive a similarity of **1.00**.

This does not mean that the sentences are identical in every linguistically relevant respect. It means that they are identical **under this representation**.


## Surface forms and lemmas produce different vector spaces

The next six documents form two small groups. The first three concern animals; the final three concern markets.

We will create one bag-of-words space from their surface forms and another from the supplied lemmas. We then compare the documents within each space.


In [ ]:
document_labels = [
    "Animals 1", "Animals 2", "Animals 3",
    "Markets 1", "Markets 2", "Markets 3",
]

documents = [
    "Dogs chase cats in the garden.",
    "A dog chased the cat through the gardens.",
    "Cats and dogs were chasing each other.",
    "Markets rose after investors bought shares.",
    "The market rises when an investor buys shares.",
    "Investors are buying shares as markets rise.",
]

pd.DataFrame({"document": document_labels, "text": documents})


In [ ]:
def vectorise_surface_forms(texts):
    vectoriser = CountVectorizer(token_pattern=r"(?u)\b[a-zA-Z]+\b")
    matrix = vectoriser.fit_transform(texts)
    return vectoriser, matrix

def vectorise_supplied_lemmas(texts):
    lemmatised_texts = [" ".join(supplied_lemmatisation(text)) for text in texts]
    vectoriser = CountVectorizer(token_pattern=r"(?u)\b[a-zA-Z]+\b")
    matrix = vectoriser.fit_transform(lemmatised_texts)
    return lemmatised_texts, vectoriser, matrix

surface_vectoriser, surface_matrix = vectorise_surface_forms(documents)
lemma_documents, lemma_vectoriser, lemma_matrix = vectorise_supplied_lemmas(documents)

print("Surface-form vocabulary size:", len(surface_vectoriser.get_feature_names_out()))
print("Lemma vocabulary size:       ", len(lemma_vectoriser.get_feature_names_out()))

pd.DataFrame({
    "document": document_labels,
    "surface text": documents,
    "supplied lemma sequence": lemma_documents,
})


In [ ]:
surface_similarity = cosine_similarity(surface_matrix)
lemma_similarity = cosine_similarity(lemma_matrix)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.8))

for ax, values, title in [
    (axes[0], surface_similarity, "Surface-form bag of words"),
    (axes[1], lemma_similarity, "Lemma bag of words"),
]:
    table = pd.DataFrame(values, index=document_labels, columns=document_labels)
    sns.heatmap(
        table,
        annot=True,
        fmt=".2f",
        cmap="YlGnBu",
        vmin=0,
        vmax=1,
        square=True,
        linewidths=0.4,
        cbar=ax is axes[1],
        ax=ax,
    )
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=45)
    ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()


### Reading the heatmaps

Make a note of:

- whether documents from the same group become more similar after lemmatisation;
- why **chase**, **chased** and **chasing** occupy separate dimensions in one representation but a shared dimension in the other;
- any similarities caused by common grammatical words rather than the subject matter;
- why neither heatmap can represent who chased whom.

The numbers changed because the representation changed. Cosine similarity did not discover a uniquely correct notion of document similarity.


# Final synthesis — One document, several computational objects

The following cell makes each stage visible for one document.


In [ ]:
chosen_document = documents[1]
chosen_tokens = simple_word_tokens(chosen_document)
chosen_lemmas = supplied_lemmatisation(chosen_document)

lemma_count_vectoriser = CountVectorizer(
    vocabulary=lemma_vectoriser.vocabulary_,
    token_pattern=r"(?u)\b[a-zA-Z]+\b",
)
chosen_vector = lemma_count_vectoriser.transform([" ".join(chosen_lemmas)])
feature_names = lemma_count_vectoriser.get_feature_names_out()
non_zero_counts = {
    feature_names[index]: int(value)
    for index, value in zip(chosen_vector.indices, chosen_vector.data)
}

print("Original text:")
print(chosen_document)
print("\nTokens:")
print(chosen_tokens)
print("\nSupplied lemmas:")
print(chosen_lemmas)
print("\nNon-zero bag-of-words coordinates:")
print(dict(sorted(non_zero_counts.items())))


## What to retain for L2

For each representation, be ready to discuss:

- what distinctions it preserves;
- what distinctions it combines;
- what it discards or leaves implicit;
- what comparisons it makes possible.

The central conclusion is:

> **Similarity depends on representation. A model cannot use distinctions that its input representation has discarded.**


# Further exploration — optional

If you have time, try one or more of the following:

- add another animal or market document and inspect its heatmap row;
- add an unrelated document and investigate any non-zero similarities;
- use `CountVectorizer`'s built-in English stop-word list to remove common words and compare the heatmap;
- alter one document's word order without changing its words and confirm that its unigram bag-of-words vector remains unchanged.

Do not interpret every change as an improvement. State what information the new representation uses and what it ignores.


In [ ]:
# Optional example: remove common English stop words and compare again.
stopword_vectoriser = CountVectorizer(
    stop_words="english",
    token_pattern=r"(?u)\b[a-zA-Z]+\b",
)
stopword_matrix = stopword_vectoriser.fit_transform(documents)
stopword_similarity = cosine_similarity(stopword_matrix)

fig, ax = plt.subplots(figsize=(7.2, 5.8))
sns.heatmap(
    pd.DataFrame(stopword_similarity, index=document_labels, columns=document_labels),
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.4,
    ax=ax,
)
ax.set_title("Optional: surface-form similarity after stop-word removal")
ax.tick_params(axis="x", rotation=45)
ax.tick_params(axis="y", rotation=0)
plt.tight_layout()
plt.show()


## Documentation used in this notebook

- [NLTK tokenisation](https://www.nltk.org/api/nltk.tokenize.html)
- [NLTK WordNet lemmatiser](https://www.nltk.org/api/nltk.stem.wordnet.html)
- [NLTK Porter stemmer](https://www.nltk.org/api/nltk.stem.PorterStemmer.html)
- [scikit-learn CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [scikit-learn cosine similarity](https://scikit-learn.org/stable/modules/metrics.html#cosine-similarity)
